# Setup

In [2]:
from manim import *
import numpy as np
from scipy.optimize import fsolve
config.media_width = "75%"
config.verbosity = "WARNING"

# colors

In [ ]:
%%manim -qh Thumbnail

class Thumbnail(Scene):
    def construct(self):
        title = Tex("Title", font_size=165)
        #title.set_color_by_gradient(*gradient_colors)
        self.add(title)

# Title

In [44]:
%%manim -qh S0

class S0(Scene):
    def construct(self):
        con = Tex(r"Continuity", font_size=110).to_edge(UP)
        top = Tex(r"Topology", font_size=110).shift(RIGHT*4.5)
        spaces = Tex(r"Topological spaces", font_size=110).to_edge(DOWN).set_color_by_gradient(*["#FFF720", "#3CD500"])

        simple = ImageMobject("continuity-simple.png").scale(0.6).shift(LEFT*2.5 + UP*0.2)

        axes = Axes(
            x_range=[-1, 10, 1],
            y_range=[-1, 6, 1],
            x_length=6,
            y_length=4,
            axis_config={"include_numbers": False}
        )
        f0 = lambda x : (0.3*x)**3 - 3* (0.3*x)**2 + 6 
        f = lambda x : f0(x) - f0(4) + 2
        graph = axes.plot(f, color=RED)

        gr = VGroup(axes, graph)

        self.add(gr[0])
        self.play(Write(con), Create(gr[1]))
        self.wait()
        self.play(gr.animate.scale(0.5).shift(UP*2.5 + RIGHT*5), Write(top), FadeIn(simple))
        self.wait()
        self.play(Write(spaces))
        self.wait()
        self.play(FadeOut(gr, con, top, simple, spaces))
        

Manim Community v0.19.0

In [45]:
%%manim -qh S1

class S1(Scene):
    def construct(self):
        axes = Axes(
            x_range=[-1, 10, 1],
            y_range=[-1, 6, 1],
            x_length=6,
            y_length=4,
            axis_config={"include_numbers": False}
        ).shift(DOWN*1.5)
        f0 = lambda x : (0.3*x)**3 - 3* (0.3*x)**2 + 6 
        f = lambda x : f0(x) - f0(4) + 2
        graph = axes.plot(f, color=GREEN)

        a = 4
        f_a = f(a)
        dot = Dot(axes.c2p(a, f(a)))
        label_a = MathTex("x").next_to(axes.c2p(a, 0), DOWN)
        label_fa = MathTex("f(x)").next_to(axes.c2p(0, f_a), LEFT)
        
        eps_color = RED
        def gen_eps_group(eps):
            eps_outline = VGroup(axes.plot(lambda x: f_a-eps, color=eps_color, x_range=[0,10]), axes.plot(lambda x : f_a+eps, color=eps_color, x_range=[0,10]))
            eps_band = axes.get_area(eps_outline[1], bounded_graph=eps_outline[0], color=eps_color, opacity=0.15)
            eps_brace = Brace(eps_band, RIGHT)
            eps_brace_label = MathTex(r"2\varepsilon").next_to(eps_brace, RIGHT)
            return VGroup(eps_outline, eps_band, eps_brace, eps_brace_label)

        delta_color = DARK_BLUE
        def gen_delta_group(delta):
            delta_outline = VGroup(Line(axes.c2p(a-delta, 0), axes.c2p(a-delta, 4.5), color=delta_color), Line(axes.c2p(a+delta, 0), axes.c2p(a+delta, 4.5), color=delta_color))
            delta_band = axes.get_area(axes.plot(lambda x : 4.5), color=delta_color, opacity=0.15, x_range=[a-delta,a+delta])
            delta_up = axes.plot(f, color=delta_color, x_range=[a-delta,a+delta])
            #delta_down = axes.plot(lambda x : 0, color=delta_color, x_range=[a-delta,a+delta])
            delta_brace = Brace(delta_band, UP)
            delta_brace_label = MathTex(r"2\delta").next_to(delta_brace, UP)
            return VGroup(delta_outline, delta_band, delta_up, delta_brace, delta_brace_label)

       
        eps_tracker = ValueTracker(1.75)
        delta_tracker = ValueTracker(1.5)
        eps_group = always_redraw(lambda: gen_eps_group(eps_tracker.get_value()))        
        delta_group = always_redraw(lambda: gen_delta_group(delta_tracker.get_value()))

        title = Tex(r"$\varepsilon$-$\delta$ definition of continuity", font_size=55).to_edge(UP).shift(UP*0.3)
        under = Underline(title)
        deff = Tex(r"$f:\mathbb{R} \to \mathbb{R}$ is continuous at $x \in \mathbb{R}$ \\ if for every $\varepsilon > 0$, there is a $\delta > 0$ such that \\ $|x-y| < \delta$ implies $|f(x) - f(y)| < \varepsilon$.")
        deff.next_to(under, DOWN)

        expla = VGroup(title, under, deff)

        graph_group = VGroup(axes, graph, label_a, label_fa, dot, eps_group, delta_group)
        
        self.play(FadeIn(title, under, axes, graph))
        self.wait()
        self.play(FadeIn(label_a, label_fa, dot))
        self.wait()
        self.play(FadeIn(deff))
        self.play(FadeIn(eps_group))
        self.wait()
        self.play(FadeIn(delta_group))
        self.wait()
        self.play(Circumscribe(delta_group[2], time_width=3))
        self.wait()
        self.play(eps_tracker.animate.set_value(0.8))
        self.wait()
        self.play(delta_tracker.animate.set_value(0.7))
        self.wait()

        self.play(FadeOut(title, deff, under), graph_group.animate.shift(UP*3))
        self.play(FadeOut(delta_group))

        rts = [fsolve(lambda x : f(x) - f_a - eps_tracker.get_value(), g)[0] for g in [3, 9]]
        rts2 = [fsolve(lambda x : f(x) - f_a + eps_tracker.get_value(), g)[0] for g in [5, 8]]
        precolor = PINK
        preimage = VGroup(Line(axes.c2p(rts[0], 0), axes.c2p(rts2[0], 0), stroke_width=13), Line(axes.c2p(rts[1], 0), axes.c2p(rts2[1], 0), stroke_width=13)).set_color(precolor)
        pretex = MathTex(r"f^{-1}(B_\varepsilon(f(x)))", r" = \{y \mid |f(x) - f(y)| < \varepsilon\}").next_to(graph_group, DOWN).shift(DOWN*0.3)
        pretex[0].set_color(precolor)

        self.play(FadeIn(preimage), Write(pretex))
        self.wait()
        neigh1 = MathTex(r"\exists \delta > 0:", r"B_\delta(x)", r"\subseteq", r"f^{-1}(B_\varepsilon(f(x)))").next_to(pretex, DOWN)
        neigh1[1].set_color(delta_color)
        neigh1[3].set_color(precolor)
        self.play(FadeIn(delta_group), Write(neigh1))
        self.wait()
        neigh2 = Tex(r"i.e. ", r"$f^{-1}(B_\varepsilon(f(x)))$", r" is a neighborhood of $x$.").next_to(neigh1, DOWN)
        neigh2[1].set_color(precolor)
        self.play(Write(neigh2))
        self.wait()
        self.play(FadeOut(pretex, neigh1, neigh2))
        conti1 = Tex(r"$f:\mathbb{R} \to \mathbb{R}$ is continuous at $x$ if and only if \\ for every $\varepsilon > 0$, $f^{-1}(B_\varepsilon(f(x)))$ is a neighborhood of $x$.", font_size=52)
        conti1.next_to(graph_group, DOWN).shift(DOWN*0.3)
        neighdef = Tex(r"($U \subseteq \mathbb{R}$ is a neighborhood of $x\in \mathbb{R}$ \\ if there is a $\delta > 0$ such that $B_\delta(x) \subseteq U$.)")
        neighdef.next_to(conti1, DOWN).shift(DOWN*0.2)
        self.play(FadeIn(conti1, neighdef))
        self.wait()
        self.play(FadeOut(neighdef, graph_group, preimage), conti1.animate.to_edge(UP))
        inclu = MathTex(r"B_\varepsilon(f(x)) \subseteq U \Rightarrow f^{-1}(B_\varepsilon(f(x))) \subseteq f^{-1}(U)").next_to(conti1, DOWN).shift(DOWN*1)
        conti2 = Tex(r"$f:\mathbb{R} \to \mathbb{R}$ is continuous at $x$ if and only if \\ for every neighborhood $U$ of $f(x)$,\\ $f^{-1}(U)$ is a neighborhood of $x$.", font_size=52)
        conti2.next_to(inclu, DOWN).shift(DOWN*1)
        self.play(FadeIn(inclu))
        self.wait()
        self.play(Write(conti2))
        self.wait()
        self.play(FadeOut(conti1, inclu), conti2.animate.move_to(ORIGIN).scale(1.2))
        self.wait()
        self.play(conti2.animate.to_edge(UP))
        topo1 = Tex(r"Definition of a ", r"topological space", " (via ", r"neighborhoods", r")", font_size=52).move_to(DOWN*0.3)
        topo1[3].set_color(ORANGE)
        topo1[1].set_color(GREEN)
        underline = Underline(topo1)
        toponeigh = Tex(r"A ", r"topological space", r" $(X, \mathcal{N})$ is a set $X$  with a function $\mathcal{N}$ \\ assigning to each $x \in X$  a collection $\mathcal{N}(x)$ of subsets of $X$,\\ called ", r"neighborhoods", r" of $x$, such that")
        toponeigh[1].set_color(GREEN)
        toponeigh[3].set_color(ORANGE)
        toponeigh.next_to(underline, DOWN)
        dots = MathTex(r"\ldots", font_size=150).next_to(toponeigh, DOWN).shift(DOWN*0.25)
        top_group = VGroup(topo1, underline, toponeigh, dots)
        
        self.play(FadeIn(topo1, underline))
        self.wait()
        self.play(FadeIn(toponeigh, dots))
        self.wait()
        self.play(FadeOut(top_group, conti2))

Manim Community v0.19.0

In [46]:
%%manim -qh S2

class S2(Scene):
    def construct(self):
        svg = SVGMobject("continuity-at-point.svg", height=8)
        subm = svg.submobjects
        
        defconti = subm[1]
        fdef = VGroup(*subm[2:7])
        topX, topY = subm[8], subm[7]
        neighU, neighfU = VGroup(*subm[9:11]), VGroup(*subm[11:18])
        px, pfx = VGroup(*subm[20:22]), VGroup(*subm[18:20])
        px.z_index = 10
        pfx.z_index = 10
        arr = ImageMobject("continuity-at-point-arrow.png")

        self.play(Create(topX))
        self.play(FadeIn(arr, fdef))
        self.play(Create(topY))
        self.wait()
        self.play(FadeIn(px))
        self.play(FadeIn(pfx))
        self.wait()
        self.play(FadeIn(neighU))
        self.wait()
        self.play(FadeIn(neighfU))
        self.wait()
        self.play(FadeIn(defconti))
        self.wait()
        self.play(FadeOut(svg[1:], arr))

Manim Community v0.19.0

In [47]:
%%manim -qh S3

class S3(Scene):
    def construct(self):
        topo1 = Tex(r"Definition of a ", r"topological space", " (via ", r"neighborhoods", r")", font_size=52).to_edge(UP)
        topo1[3].set_color(ORANGE)
        topo1[1].set_color(GREEN)
        underline = Underline(topo1)
        toponeigh = Tex(r"A ", r"topological space", r" $(X, \mathcal{N})$ is a set $X$  with a function $\mathcal{N}$ \\ assigning to each $x \in X$  a collection $\mathcal{N}(x)$ of subsets of $X$,\\ called ", r"neighborhoods", r" of $x$, such that")
        toponeigh[1].set_color(GREEN)
        toponeigh[3].set_color(ORANGE)
        toponeigh.next_to(underline, DOWN)
        dots = MathTex(r"\ldots", font_size=150).next_to(toponeigh, DOWN).shift(DOWN*0.25)
        top_group = VGroup(topo1, underline, toponeigh)
        self.play(FadeIn(top_group, dots))
        self.wait()

        naxioms = Tex(r"(i) $X \in \mathcal{N}(x)$ for every $x \in X$. \\", 
            r"(ii) Let $U \subseteq X$, then $U \in \mathcal{N}(x)$ if and only if \\", r"there is an open $V \subseteq X$ such that $x \in V \subseteq U$. \\",
            r"(iii) If $U, V \in \mathcal{N}(x)$, then $U \cap V \in \mathcal{N}(x)$.")
        naxioms.arrange(DOWN, aligned_edge=LEFT, buff=0.25)
        naxioms[2:].shift(UP*0.1)
        naxioms.next_to(toponeigh, DOWN)
        self.play(ReplacementTransform(dots, naxioms[0]))
        self.wait()

        numberline = NumberLine(
            x_range=[-10, 10, 2],
            length=10,
            include_numbers=False,
            label_direction=UP,
        )
        neighR = Tex(r"$A \subseteq \mathbb{R}$ is a neighborhood of $x \in \mathbb{R}$ \\ if $\exists \delta > 0$ such that $(x-\delta, x+\delta) \subseteq A.$")
        neighR.to_edge(DOWN).shift(DOWN*0.2)
        numberline.next_to(neighR, UP)
        self.play(FadeIn(neighR))
        self.wait()
        openInter = VGroup(MathTex(r"(", font_size=70), MathTex(r")", font_size=70)).move_to(numberline).set_color(RED)
        openInter[0].shift(LEFT*2)
        openInter[1].shift(RIGHT*2)
        openInter.add(Rectangle(fill_color=RED, fill_opacity=0.5, height=0.35, width=4.1, stroke_width=0))
        openInter[2].move_to(numberline)
        self.play(FadeIn(openInter, numberline))
        self.wait()
        dot = Dot().move_to(numberline)
        self.play(FadeIn(dot))
        self.wait()
        self.play(dot.animate.shift(LEFT*1.5))
        self.wait()
        self.play(dot.animate.shift(RIGHT*3.25))
        self.wait()
        
        opendef = Tex(r"A set $A$ is called ", r"open", r" if its a neighborhood of every $x \in A$.").to_edge(DOWN)
        opendef[1].set_color(RED)
        self.play(ReplacementTransform(neighR, opendef))
        self.wait()
        self.play(FadeOut(numberline, openInter, dot), Write(naxioms[1:3]))
        self.wait()
        self.play(Write(naxioms[3]))
        self.wait()
        self.play(FadeOut(top_group, naxioms, opendef))


Manim Community v0.19.0

In [48]:
%%manim -qh S4

class S4(Scene):
    def construct(self):
        svg = SVGMobject("continuity-at-point.svg", height=8)
        subm = svg.submobjects
        
        defconti = subm[1]
        fdef = VGroup(*subm[2:7])
        topX, topY = subm[8], subm[7]
        neighU, neighfU = VGroup(*subm[9:11]), VGroup(*subm[11:18])
        px, pfx = VGroup(*subm[20:22]), VGroup(*subm[18:20])
        px.z_index = 10
        pfx.z_index = 10
        arr = ImageMobject("continuity-at-point-arrow.png")
        contiev = Tex(r"$U \subseteq Y$ is open $\Rightarrow$ $f^{-1}(U) \subseteq X$ is open", font_size=70).to_edge(DOWN).shift(DOWN*0.3)

        self.play(FadeIn(svg[1:], arr))
        self.wait()
        self.play(px.animate.shift(DOWN), pfx.animate.shift(RIGHT*1.3 + DOWN*0.5))
        self.wait()
        self.play(FadeOut(px, pfx, defconti))
        self.play(Write(contiev))
        self.wait()
        self.play(FadeOut(svg[2:18], arr, contiev))

Manim Community v0.19.0

In [49]:
%%manim -qh S5

class S5(Scene):
    def construct(self):
        topo1 = Tex(r"Definition of a ", r"topological space", " (via ", r"open sets", r")", font_size=52).to_edge(UP)
        topo1[3].set_color(RED)
        topo1[1].set_color(GREEN)
        underline = Underline(topo1)
        toponeigh = Tex(r"A", r" topological space", r" $(X, \mathcal{T}_X)$ is a set $X$ with a collection \\ of subsets $\mathcal{T}_X \subset \mathcal{P}(X)$, called ", r"open sets", r", such that")
        toponeigh[1].set_color(GREEN)
        toponeigh[3].set_color(RED)
        toponeigh.next_to(underline, DOWN)
        top_group = VGroup(topo1, underline, toponeigh)
        
        naxioms = Tex(r"(i) $\emptyset$ and $X$ are open, \\", 
            r"(ii) if $(U_i)_{i\in I}$ are open, then $\bigcup_{i\in I} U_i$ is open, \\",
            r"(iii) if $U,V$ are open, then $U \cap V$ is open.")
        naxioms.arrange(DOWN, aligned_edge=LEFT, buff=0.25)
        naxioms[2:].shift(UP*0.1)
        naxioms.next_to(toponeigh, DOWN)

        topon = Tex(r"Definition of a ", r"topological space", " (via ", r"neighborhoods", r")", font_size=52).to_edge(DOWN)
        topon[3].set_color(ORANGE)
        topon[1].set_color(GREEN)
        under2 = Underline(topon)
        arrs = VGroup(Arrow(start = DOWN*0.5, end = DOWN*3.1).shift(LEFT*0.75), Arrow(end = DOWN*0.5, start = DOWN*3.1).shift(RIGHT*0.75))
        # conver2 = Tex(r"$U \subseteq X$ is open if \\ its a neighborhood \\ of every $x \in U$.", font_size=40).next_to(arrs[1], RIGHT)
        conver2 = MathTex(r"\mathcal{T}_X := \{U \subseteq X \mid \forall x \in U : \\ U \in \mathcal{N}(x)\}", font_size=40).next_to(arrs[1], RIGHT)
        # conver1 = Tex(r"$V \subseteq X$ is a neighborhood of $x\in X$ \\ if there exists an open $U \subseteq X$ \\ such that $x \in U \subseteq V$.", font_size=40).next_to(arrs[0], LEFT)
        conver1 = MathTex(r"\mathcal{N}(x) := \{V \subseteq X \mid  \exists U \in \mathcal{T}_X : \\ x \in U \subseteq V\}", font_size=40).next_to(arrs[0], LEFT)

        conver = VGroup(arrs[0], conver1, arrs[1], conver2)

        self.play(FadeIn(top_group))
        self.wait()
        self.play(Write(naxioms[0]))
        self.wait()

        svg = SVGMobject("union-topo.svg", height=3).shift(DOWN*2.9)
        subm = svg.submobjects
        large = subm[0]
        points = VGroup(subm[2], subm[4], subm[6], subm[8], subm[10])
        uis = VGroup(subm[1], subm[3], subm[5], subm[7], subm[9])

        ulabel = MathTex(r"U", font_size=70).set_color(RED).next_to(large, LEFT).shift(UP*1.5)
        unitex = Tex(r"$\forall x\in U: \exists $ open ", r"$U_x$", r" such that $x \in U_x \subseteq U$", r"$\;\Rightarrow$ $U$ is open")
        unitex[1].set_color(YELLOW)
        unitex.to_edge(DOWN).shift(DOWN*0.2)
        svg.next_to(unitex, UP)
        self.play(FadeIn(large, ulabel))
        self.wait()
        self.play(FadeIn(points), Write(unitex[0:-1]))
        self.wait()
        self.play(FadeIn(uis))
        self.wait()

        unitex2 = MathTex(r"U = \bigcup_{x \in U} U_x", font_size=60).next_to(svg, RIGHT).shift(RIGHT*0.5)
        self.play(FadeIn(unitex[-1]), Write(unitex2))
        self.wait()
        self.play(FadeOut(svg, unitex, unitex2, ulabel))
        self.play(Write(naxioms[1]))
        self.wait()
        self.play(Write(naxioms[2]))
        self.wait()

        self.play(FadeIn(conver, topon, under2))
        self.wait()
        self.play(FadeOut(conver, topon, under2))
        
        
        #self.add(top_group, naxioms, topon, under2, conver)
        #self.wait()

        intersections = MathTex(r"\bigcap_{n \ge 1} \left(-\frac{1}{n}, \frac{1}{n} \right) = \{0\}").shift(DOWN*2.5)
        self.play(Write(intersections))
        self.wait()
        self.play(FadeOut(intersections, naxioms, top_group, naxioms))
        #self.play(FadeOut(conver, topon, under2))
        #self.play(Write(intersections))
        #self.wait()

Manim Community v0.19.0

[10/17/25 21:59:11] WARNING  Attempted adding some Mobject as a child more than once, this is not    ]8;id=929984;file://C:\GitHub\projects\.venv\Lib\site-packages\manim\mobject\mobject.py\mobject.py]8;;\:]8;id=781237;file://C:\GitHub\projects\.venv\Lib\site-packages\manim\mobject\mobject.py#507\507]8;;\
                             possible. Repetitions are ignored.                                                    

In [11]:
%%manim -qh S6

import random

class S6(Scene):
    def construct(self):
        title = Tex(r"Examples of ", r"topological spaces").to_edge(UP)
        title[1].set_color(GREEN)
        under = Underline(title)

        real = Tex(r"The real numbers $\mathbb{R}$, where $U \subseteq \mathbb{R}$ is open \\ if and  only if  $\forall x\in U\; \exists \varepsilon > 0 : (x-\varepsilon,x+\varepsilon) \subseteq U$")
        real.next_to(under, DOWN).shift(DOWN*0.5)
        
        metric = Tex(r"Any metric space $(X,d)$, where $U \subseteq X$ is open \\ if and only if $\forall x\in U\; \exists \varepsilon > 0 : B_\varepsilon(x) \subseteq U$")
        metric.shift(DOWN*0.5)
        
        self.play(FadeIn(title, under))
        self.wait()
        self.play(Write(real))
        self.wait()
        self.play(Write(metric))
        self.wait()

        stp = Tex(r"Let $X$ be any set.").next_to(under, DOWN)
        trivial = Tex(r"Trivial topology $\{\emptyset, X\}$").move_to(UP*1.5 + LEFT*3.5)
        discrete = Tex(r"Discrete topology $\mathcal{P}(X)$").move_to(UP*1.5 + RIGHT*3.5)
        divide = Line(start=UP*1.75, end=DOWN*3.75, buff=0)

        vis1 = VGroup()
        for i in range(15):
            vis1.add(Dot().move_to(DOWN*random.uniform(0.9, 1.1) + LEFT*random.uniform(0.9, 1.1)).shift(LEFT*2.75))

        vis2 = VGroup(
            Dot().move_to(DOWN*1 + RIGHT*3.45), Dot().move_to(DOWN*2.2 + RIGHT*2), Dot().move_to(DOWN*3 + RIGHT*4.5),
            Dot().move_to(RIGHT*5), Dot().move_to(UP*0.25 + RIGHT*1.7)
        )
        
        self.play(FadeOut(real, metric))
        self.play(FadeIn(stp))
        self.wait()
        self.play(FadeIn(trivial))
        self.wait()
        self.play(FadeIn(vis1))
        self.wait()
        self.play(FadeIn(discrete, divide))
        self.wait()
        self.play(FadeIn(vis2))
        self.wait()
        self.play(FadeOut(stp, trivial, discrete, divide, vis1, vis2, title, under))
        

Manim Community v0.19.0

In [54]:
%%manim -qh S7

class S7(Scene):
    def construct(self):
        quest = Tex(r"Do theorems from real analysis about continuity \\ generalize to ", r"topological spaces", r"?").to_edge(UP)
        quest[1].set_color(GREEN)
        
        axes = Axes(
            x_range=[-1, 10, 1],
            y_range=[-1, 6, 1],
            x_length=6,
            y_length=4,
            axis_config={"include_numbers": False}
        )
        f0 = lambda x : (0.3*x)**3 - 3* (0.3*x)**2 + 6 
        f = lambda x : f0(x) - f0(4) + 2
        graph = axes.plot(f, color=RED)

        ivt = Tex(r"Intermediate value theorem").move_to(LEFT*3.75 + DOWN*3)
        ext = Tex(r"Extreme value theorem").move_to(RIGHT*3.75 + DOWN*3)

        self.play(Write(quest), FadeIn(axes), Create(graph))
        self.wait()
        self.play(FadeIn(ivt, ext))
        self.wait()
        self.play(FadeOut(axes, graph, quest, ivt, ext))

Manim Community v0.19.0

In [51]:
%%manim -qh S8

class S8(Scene):
    def construct(self):
        title = Tex(r"Intermediate value thorem", font_size=60).to_edge(UP)
        under = Underline(title)
        inter = Tex(r"Let $f:[a,b] \to \mathbb{R}$ be continuous, then $[f(a); f(b)] \subseteq f([a,b])$, \\ where $[x;y] := \{x + \lambda (y-x) \mid 0 \le\lambda \le 1\}$.")
        inter.next_to(under, DOWN)
        arr1 = Arrow(start=UP, end=DOWN).next_to(inter, DOWN)
        conn1 = Tex(r"Let $f:X \to \mathbb{R}$ be continuous, $X$ ", r"connected", r" and $a,b \in X$, \\ then $[f(a); f(b)] \subseteq f(X)$.")
        conn1[1].set_color(BLUE)
        conn2 = Tex(r"Let $f:X \to Y$ be continuous and $X$ ", r"connected", r", \\ then $f(X)$ is ", r"connected", r".")
        conn2[1].set_color(BLUE)
        conn2[3].set_color(BLUE)
        conn1.next_to(arr1, DOWN)
        conn2.next_to(conn1, DOWN).shift(DOWN*0.5)
        self.play(FadeIn(title, under))
        self.play(Write(inter))
        self.wait()
        self.play(GrowArrow(arr1))
        self.play(FadeIn(conn1))
        self.wait()
        self.play(FadeIn(conn2))
        self.wait()
        self.play(FadeOut(title, under, inter, arr1, conn1, conn2))


Manim Community v0.19.0

In [52]:
%%manim -qh S9

class S9(Scene):
    def construct(self):
        title = Tex(r"Extreme value theorem", font_size=60).to_edge(UP)
        under = Underline(title)
        inter = Tex(r"Let $f:[a,b] \to \mathbb{R}$ be continuous, \\ then $f$ attains its maximum and minimum.")
        inter.next_to(under, DOWN)
        arr1 = Arrow(start=UP, end=DOWN).next_to(inter, DOWN)
        conn1 = Tex(r"Let $f:X \to \mathbb{R}$ be continuous, and $X$ ", r"compact", r", \\ then $f$ attains its maximum and minimum.")
        conn1[1].set_color(RED)
        conn2 = Tex(r"Let $f:X \to Y$ be continuous and $X$ ", r"compact", r", \\ then $f(X)$ is ", r"compact", r".")
        conn2[1].set_color(RED)
        conn2[3].set_color(RED)
        conn1.next_to(arr1, DOWN)
        conn2.next_to(conn1, DOWN).shift(DOWN*0.5)
        self.play(FadeIn(title, under))
        self.play(Write(inter))
        self.wait()
        self.play(GrowArrow(arr1))
        self.play(FadeIn(conn1))
        self.wait()
        self.play(FadeIn(conn2))
        self.wait()
        self.play(FadeOut(title, under, inter, arr1, conn1, conn2))

Manim Community v0.19.0

In [10]:
%%manim -qm S10

class S10(Scene):
    def construct(self):
        title = Tex(r"How to compare two topological spaces?").to_edge(UP)
        under = Underline(title)

        top1 = VGroup(Tex(r"1").shift(UP*0.5+LEFT*0.5), Tex(r"2").shift(UP*0.5+RIGHT*0.5), Tex(r"3").shift(DOWN*0.5+LEFT*0.5), Tex(r"4").shift(DOWN*0.5+RIGHT*0.5))
        top2 = VGroup(Tex(r"a").shift(UP*0.5+LEFT*0.5), Tex(r"b").shift(UP*0.5+RIGHT*0.5), Tex(r"c").shift(DOWN*0.5+LEFT*0.5), Tex(r"d").shift(DOWN*0.5+RIGHT*0.5))
        top1lab = 
        
        self.add(title, under, top1)
        self.wait()

Manim Community v0.19.0